# SpaceJEPA v3 · All results in one PDF

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/jepa-v3/StochasticEntrySim/ReEntryAI_JEPA_v3_resultspdf.ipynb)

**Runtime: standard CPU** (it opens that way; nothing here needs a GPU or High-RAM). It takes about
5–10 min, most of it reading from Drive, and releases the runtime by itself when it finishes or a
cell fails (set `AUTO_DISCONNECT = False` in the setup cell to keep it).

It reads what the four v3 notebooks saved in **`MyDrive/ReEntryAI_jepa_v3/`** and writes one PDF,
**`06_report/SpaceJEPA_v3_full_results.pdf`**, with:

| Section | From | What |
|---|---|---|
| 0. What is in the study folder | all | files and MB per stage, what is finished |
| 1. Settings | `00_study/` | every frozen setting, git commit, environment |
| 2. Test set | notebook 0 | the 64 cases, feasibility per atmosphere, entry ranges, F01, F02 |
| 3. Training | notebook 1 | per model and seed: checkpoint notes, final losses, latent health, training curves |
| 4. Flights | notebook 2 | flights per arm, how they ended, first look per atmosphere, timing |
| 5. Closed loop | notebook 3 | criteria C1–C7 + Bridge (T05), T01–T08, F10–F21 |
| 6. Open loop | notebook 3 | F30, F31, T09, T10, along-track bias |
| Appendix | all | the long tables (full T01, T01b, T11, the test set), anything not shown above, every caption, a folder index |

Nothing is recomputed: every number, table and figure is read from the files, with captions and
data sources from `05_analysis/captions.json`. With a partly finished study it shows what exists
and names what is missing. It is a separate file from notebook 3's shorter
`SpaceJEPA_v3_report.pdf`.

## 0. Setup

On Colab this clones the `jepa-v3` branch, installs `pymsis` and `reportlab`, and mounts Google Drive.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "jepa-v3"
MODE = os.environ.get("JEPA_V3_MODE", "full")     # "full": the study | "smoke": the smoke-mode study folder
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis reportlab
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    STUDY_ROOT = "/content/drive/MyDrive/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
else:
    STUDY_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")

from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
from jepa import study_v3 as S, results_pdf_v3 as RP
L = S.Layout(STUDY_ROOT)
assert L.root.exists(), f"no study folder at {STUDY_ROOT}: run the v3 notebooks first (or check MODE)"
print(f"MODE={MODE}  study folder={STUDY_ROOT}")

# ---- Auto-disconnect: stop paying for the runtime once the work is done or a cell fails ----
AUTO_DISCONNECT = IN_COLAB      # set to False to keep the runtime (e.g. to explore results by hand)

def release_runtime(reason):
    # Flush Drive, then release the Colab runtime so it stops using compute units.
    if not (IN_COLAB and AUTO_DISCONNECT):
        print(f"[auto-disconnect off] {reason}")
        return
    print(f"[auto-disconnect] {reason}. Flushing Drive and releasing the runtime; outputs stay visible.", flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("  Drive flush:", e)
    from google.colab import runtime
    runtime.unassign()

def _release_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is not None and not isinstance(err, KeyboardInterrupt):     # pressing stop is not a failure
        release_runtime(f"a cell failed ({type(err).__name__}: {str(err)[:200]})")

_ip = get_ipython()
for _cb in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_release_on_error":
        _ip.events.unregister("post_run_cell", _cb)
_ip.events.register("post_run_cell", _release_on_error)
print("auto-disconnect on: the runtime is released when the last cell finishes or a cell fails" if AUTO_DISCONNECT
      else "auto-disconnect off (not on Colab, or switched off): the runtime stays up until you disconnect it")

## 1. Build the PDF

In [ ]:
PDF = RP.build(STUDY_ROOT)

## 2. Refresh the folder index

Rebuilds `00_study/manifest.csv` and `README.md` so they list the new PDF and everything notebook 3
wrote.

In [ ]:
S.update_manifest(L)
print(open(L.study / "README.md").read())
print("PDF:", PDF)

## Done: release the runtime

Everything this notebook made is already on Drive. This cell flushes Drive and releases the Colab
runtime, so it stops using compute units (a failed cell does the same). To keep the runtime, set
`AUTO_DISCONNECT = False` in the setup cell before running.

In [ ]:
release_runtime("all cells finished")